# Batch SAM3 tree segmentation in Google Colab

Generate candidate tree masks for the natural and plotted DBH datasets using
[Meta's official SAM3 repository](https://github.com/facebookresearch/sam3).
**Final masks are already provided in `data/Segmentation`; this notebook is only for regenerating them or processing new images.**

Select **Runtime → Change runtime type → GPU** (Python 3.12+). Copy the entire
`preprocessing/` folder and the required `data/rawdata/rgb_naturalField/` and/or
`data/rawdata/rgb_plottedField/` folders into `MyDrive/DBH-pipeline/`.
Both `.py` and `.ipynb` work in Colab: this notebook provides setup and runs the
shared `.py` implementation, so there is only one segmentation algorithm to maintain.

## 1. Mount Drive and set the dataset

Adjust `PROJECT_ROOT` to your Drive location. `LIMIT = 0` processes every image;
use a small positive value for a trial. Existing matching candidates are kept
unless `OVERWRITE = True`. Reuse the same settings when resuming a batch.

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/DBH-pipeline")
DATASET = "all"                 # "natural", "plot", or "all"
PROMPT = "tree trunk"
CONFIDENCE = 0.3
MAX_SIDE = 1024                 # Pre-inference resize; saved masks use original RGB dimensions.
PRECISION = "auto"              # bfloat16 on supported GPUs, otherwise float32.
LIMIT = 0
OVERWRITE = False
CHECKPOINT = None               # Or a local/Drive path to the original SAM3 sam3.pt checkpoint.

SCRIPT = PROJECT_ROOT / "preprocessing/sam3_batch_segment.py"
REQUIREMENTS = PROJECT_ROOT / "preprocessing/requirements.txt"
if not SCRIPT.is_file() or not REQUIREMENTS.is_file():
    raise FileNotFoundError("Copy preprocessing/ to PROJECT_ROOT before continuing.")
if DATASET not in {"natural", "plot", "all"}:
    raise ValueError("DATASET must be natural, plot, or all.")

def run_streamed(command):
    # A fresh process uses installed packages without keeping old Colab imports
    # or GPU tensors alive. Argument lists safely handle paths containing spaces.
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f"Command exited with code {status}; see the output above.")

## 2. Install SAM3 and check the GPU

Install from the official repository, including its notebook dependencies.
A compatible preinstalled CUDA/PyTorch build is reused; otherwise install the
build specified by the [official installation instructions](https://github.com/facebookresearch/sam3#installation).
Installation and inference run in fresh processes, avoiding stale imports after package changes.

In [ ]:
if sys.version_info < (3, 12):
    raise RuntimeError("Choose a Colab runtime with Python 3.12 or newer.")

# Check package versions in a separate process, without importing Torch into this kernel.
torch_check = """
import torch, torchvision
from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version('2.7')
assert torch.version.cuda and Version(torch.version.cuda) >= Version('12.6')
"""
probe = subprocess.run([sys.executable, "-c", torch_check], capture_output=True, text=True)
if probe.returncode:
    run_streamed([sys.executable, "-m", "pip", "install", "torch==2.10.0", "torchvision",
                  "--index-url", "https://download.pytorch.org/whl/cu128"])

# SAM3 still imports pkg_resources, which setuptools removed in version 82.
# https://setuptools.pypa.io/en/stable/deprecated/pkg_resources.html
run_streamed([sys.executable, "-m", "pip", "install", "setuptools<82",
              "sam3[notebooks] @ git+https://github.com/facebookresearch/sam3.git",
              "-r", str(REQUIREMENTS)])

gpu_check = torch_check + """
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab, then rerun setup.'
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('SAM3 imports succeeded; weights have not been loaded yet.')
"""
run_streamed([sys.executable, "-u", "-c", gpu_check])

## 3. Authenticate for model weights

First obtain access to [facebook/sam3 on Hugging Face](https://huggingface.co/facebook/sam3).
Enter your read token in the hidden prompt below; never paste it into notebook code.
This step is skipped when `CHECKPOINT` points to an existing `sam3.pt` file.

In [ ]:
if CHECKPOINT is None:
    from getpass import getpass
    from huggingface_hub import login
    login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
elif not Path(CHECKPOINT).is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT}")

## 4. Check input paths, then run the batch

The first cell only lists input counts. The next cell loads the model once and
segments each RGB image. Candidate archives and a CSV report are written to Drive,
so completed images can be reused after a disconnected session.

In [ ]:
command = [sys.executable, "-u", str(SCRIPT), "--project-root", str(PROJECT_ROOT),
           "--dataset", DATASET, "--prompt", PROMPT, "--confidence", str(CONFIDENCE),
           "--max-side", str(MAX_SIDE), "--precision", PRECISION, "--limit", str(LIMIT)]
if CHECKPOINT is not None:
    command += ["--checkpoint", str(CHECKPOINT)]
if OVERWRITE:
    command.append("--overwrite")
run_streamed(command + ["--dry-run"])

In [ ]:
# Actual GPU inference: run after confirming the paths/counts above.
# Nonzero exit status means some images need review in segmentation_report.csv.
run_streamed(command)

## 5. Select the target trees locally

Copy `output/sam3_preprocessing/` from Drive into the local `DBH-pipeline/output/`,
using the same RGB files. From the local `DBH-pipeline` folder run:

```sh
python -m pip install -r preprocessing/requirements.txt
python preprocessing/select_target_tree.py --dataset all --save-overlays
```

Choose `natural` or `plot` instead of `all` if you generated only one dataset.
Selection reads `data/basePoints_natural.csv` and `data/basePoints_plotted.csv`.
Add `--overwrite` to replace existing supplied masks or to reselect after changing points.

Each full-resolution candidate is stored separately in
`output/sam3_preprocessing/{natural,plot}/candidates/<uniqueName>_candidates.npz`.
Overlapping and disconnected regions are preserved. The selected final mask is an
8-bit PNG (tree 255, background 0), named `<uniqueName>_selected_mask.png`, in
`data/Segmentation/MaskForNaturalDataset/` or `data/Segmentation/MaskForPlottedDataset/`.
Reports and optional overlays remain under `output/sam3_preprocessing/`.